# REBUS vs Baseline

Focused comparison between the baseline and the current best downloaded REBUS variant.

The notebook deliberately avoids REBUS-only diagnostics for the baseline comparison. It uses shared readouts instead: headline metrics, outcome mix, scenario-family outcomes, partner-style outcomes, action/context patterns, belief confidence, delay, regret, and degradation patterns.


## 1. Imports and Configuration


In [ ]:
from pathlib import Path
import json
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

REPO_ROOT_OVERRIDE = None  # e.g. Path('/Users/stephenbeale/Projects/ToM_AI_Research_Team')
HARDCODED_REPO_ROOT = Path('/Users/stephenbeale/Projects/ToM_AI_Research_Team')

candidate_roots = []
if REPO_ROOT_OVERRIDE is not None:
    candidate_roots.append(Path(REPO_ROOT_OVERRIDE).expanduser())
for candidate in [Path.cwd(), Path.cwd().parent, HARDCODED_REPO_ROOT]:
    if candidate not in candidate_roots:
        candidate_roots.append(candidate)

REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'logs' / 'modal').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    REPO_ROOT = Path(REPO_ROOT_OVERRIDE).expanduser() if REPO_ROOT_OVERRIDE is not None else HARDCODED_REPO_ROOT

LOG_ROOT = REPO_ROOT / 'logs' / 'modal'
TARGET_TOTAL = 140000
SEEDS = [7, 11, 17, 23, 29]

BASELINE_SPEC = (
    'auxhead_clear_baseline',
    'baseline',
    LOG_ROOT / 'auxhead-clear-20260416' / 'by-seed',
)

REBUS_CANDIDATE_SPECS = [
    (
        'rebus_hybrid_explicit_mask',
        'REBUS hybrid explicit-mask',
        LOG_ROOT / 'auxhead-clear-rebus-reflective-hybrid-explicit-mask-20260621' / 'auxhead-clear-rebus-reflective-hybrid-explicit-mask-20260621',
    ),
    (
        'rebus_explicit_resolution',
        'REBUS explicit resolution',
        LOG_ROOT / 'auxhead-clear-rebus-explicit-resolution-20260621' / 'auxhead-clear-rebus-explicit-resolution-20260621',
    ),
]

RUN_SPECS = [BASELINE_SPEC] + REBUS_CANDIDATE_SPECS
RUN_ROOTS = {key: root for key, _, root in RUN_SPECS}
RUN_LABELS = {key: label for key, label, _ in RUN_SPECS}
RUN_ORDER = [key for key, _, _ in RUN_SPECS]

METRIC_COLUMNS = [
    'AmbiguityEfficiency',
    'AverageDelay',
    'CollisionRate',
    'CoordinationEfficiency',
    'DeadlockRate',
    'IntentionPredictionF1',
    'StrategySwitchAccuracy',
    'SuccessRate',
    'ToMCoordScore',
]

OUTCOME_COLUMNS = ['SuccessRate', 'CollisionRate', 'DeadlockRate']

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)

print('repo_root=', REPO_ROOT, 'exists=', REPO_ROOT.exists())
print('log_root=', LOG_ROOT, 'exists=', LOG_ROOT.exists())
if LOG_ROOT.exists():
    print('available modal log folders:')
    for child in sorted(x.name for x in LOG_ROOT.iterdir() if x.is_dir()):
        print(' -', child)


## 2. Load Baseline and REBUS Candidate Runs

The old baseline run stores headline metrics in `stdout.log`, not in `run_summary.json`, so the loader checks both places.


In [ ]:
def _safe_load_json(path: Path):
    if path is None:
        return None
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        return None


def _extract_json_after_marker(text: str, marker: str):
    start = text.find(marker)
    if start < 0:
        return None
    i = text.find('{', start + len(marker))
    if i < 0:
        return None
    depth = 0
    in_string = False
    escape = False
    for j in range(i, len(text)):
        ch = text[j]
        if in_string:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[i:j + 1])
                    except json.JSONDecodeError:
                        return None
    return None


def _load_marker_json_from_stdout(run_dir: Path, marker: str):
    stdout_path = run_dir / 'stdout.log'
    if not stdout_path.exists():
        return None
    return _extract_json_after_marker(stdout_path.read_text(errors='replace'), marker)


def find_run_dirs(root: Path, target_total: int = TARGET_TOTAL):
    if not root.exists():
        return []
    return sorted({p.parent for p in root.glob(f'**/target-{target_total}/run_summary.json')}, key=str)


def find_analysis_json(run_dir: Path):
    files = sorted((run_dir / 'analysis').glob('choice-analysis-*.json'))
    return files[-1] if files else None


def load_runs():
    rows = []
    analysis_payloads = {}
    missing = []
    for variant, label, root in RUN_SPECS:
        run_dirs = find_run_dirs(root)
        if not run_dirs:
            missing.append((variant, root))
            continue
        for run_dir in run_dirs:
            summary = _safe_load_json(run_dir / 'run_summary.json') or {}
            status = _safe_load_json(run_dir / 'run_status.json') or {}
            progress = _safe_load_json(run_dir / 'progress.json') or {}
            seed = summary.get('seed') or status.get('seed') or progress.get('seed')
            seed = int(seed) if seed is not None else None

            eval_metrics = summary.get('eval_metrics') or _load_marker_json_from_stdout(run_dir, 'eval_metrics=') or {}

            analysis_path = find_analysis_json(run_dir)
            analysis = _safe_load_json(analysis_path) if analysis_path else None
            if analysis is None:
                analysis = _load_marker_json_from_stdout(run_dir, 'choice_context_analysis=')
            if analysis is not None:
                analysis_payloads[(variant, seed)] = analysis

            row = {
                'variant': variant,
                'variant_label': label,
                'seed': seed,
                'run_dir': str(run_dir),
                'analysis_json': str(analysis_path) if analysis_path else ('stdout.log' if analysis is not None else None),
                'target_total_episodes': summary.get('target_total_episodes') or progress.get('completed_total_episodes'),
                'completed_total_episodes': status.get('completed_total_episodes') or progress.get('completed_total_episodes'),
                'returncode': summary.get('returncode'),
                'state': status.get('state'),
            }
            for col in METRIC_COLUMNS:
                row[col] = eval_metrics.get(col)
            rows.append(row)
    run_df = pd.DataFrame(rows)
    if not run_df.empty:
        run_df['variant'] = pd.Categorical(run_df['variant'], categories=RUN_ORDER, ordered=True)
        run_df = run_df.sort_values(['variant', 'seed']).reset_index(drop=True)
    return run_df, analysis_payloads, missing

run_df, analysis_payloads, missing_roots = load_runs()
print('loaded_runs=', len(run_df), 'loaded_analysis_payloads=', len(analysis_payloads))
if missing_roots:
    print('missing_or_not_yet_downloaded:')
    for variant, root in missing_roots:
        print(' -', variant, root, '| exists=', root.exists())

if run_df.empty:
    print('No runs loaded. Check LOG_ROOT and downloaded folder names.')
else:
    display(run_df)


## 3. Select Current REBUS Variant

The REBUS comparison variant is selected by mean `ToMCoordScore` from downloaded REBUS candidates. At present this should choose `REBUS hybrid explicit-mask`. If the resolution run later beats it after download, the notebook will switch automatically.


In [ ]:
metric_df = run_df.dropna(subset=['seed']).copy() if not run_df.empty else pd.DataFrame()
for col in METRIC_COLUMNS:
    if col in metric_df:
        metric_df[col] = pd.to_numeric(metric_df[col], errors='coerce')

candidate_keys = [key for key, _, _ in REBUS_CANDIDATE_SPECS]
candidate_metrics = metric_df[metric_df['variant'].astype(str).isin(candidate_keys)].dropna(subset=['ToMCoordScore'])
if candidate_metrics.empty:
    REBUS_VARIANT = 'rebus_hybrid_explicit_mask'
    print('No REBUS candidate metrics found yet; defaulting to', RUN_LABELS[REBUS_VARIANT])
else:
    rebus_scores = candidate_metrics.groupby('variant', observed=True)['ToMCoordScore'].mean().sort_values(ascending=False)
    REBUS_VARIANT = str(rebus_scores.index[0])
    print('REBUS candidates by mean ToMCoordScore:')
    display(rebus_scores.rename('mean_ToMCoordScore').to_frame())

BASELINE_VARIANT = 'auxhead_clear_baseline'
COMPARISON_VARIANTS = [BASELINE_VARIANT, REBUS_VARIANT]
print('baseline=', RUN_LABELS[BASELINE_VARIANT])
print('selected REBUS=', RUN_LABELS.get(REBUS_VARIANT, REBUS_VARIANT))

comparison_df = metric_df[metric_df['variant'].astype(str).isin(COMPARISON_VARIANTS)].copy()
comparison_df['variant_label'] = comparison_df['variant'].astype(str).map(RUN_LABELS)
display(comparison_df[['variant_label', 'seed'] + METRIC_COLUMNS])


## 4. Headline Score Comparison


In [ ]:
if comparison_df.empty:
    print('No comparable metrics loaded.')
else:
    mean_metrics = comparison_df.groupby('variant_label', observed=True)[METRIC_COLUMNS].mean()
    sd_metrics = comparison_df.groupby('variant_label', observed=True)[METRIC_COLUMNS].std()
    display(mean_metrics)
    print('SD')
    display(sd_metrics)

    plot_cols = ['ToMCoordScore', 'SuccessRate', 'CollisionRate', 'DeadlockRate', 'IntentionPredictionF1', 'StrategySwitchAccuracy']
    plot_cols = [c for c in plot_cols if c in mean_metrics]
    ax = mean_metrics[plot_cols].T.plot(kind='bar', figsize=(12, 5), rot=30)
    ax.set_title('REBUS vs baseline: headline metrics')
    ax.set_ylabel('mean score / rate')
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()

    baseline_label = RUN_LABELS[BASELINE_VARIANT]
    rebus_label = RUN_LABELS[REBUS_VARIANT]
    if baseline_label in mean_metrics.index and rebus_label in mean_metrics.index:
        deltas = (mean_metrics.loc[rebus_label] - mean_metrics.loc[baseline_label]).rename('REBUS_minus_baseline')
        display(deltas.to_frame())
        favorable = pd.Series({
            'success gain': deltas.get('SuccessRate', np.nan),
            'collision reduction': -deltas.get('CollisionRate', np.nan),
            'deadlock reduction': -deltas.get('DeadlockRate', np.nan),
            'ToMCoordScore gain': deltas.get('ToMCoordScore', np.nan),
            'F1 gain': deltas.get('IntentionPredictionF1', np.nan),
        }).dropna()
        ax = favorable.plot(kind='bar', figsize=(9, 4), color=['#2ca02c', '#d62728', '#ff7f0e', '#1f77b4', '#9467bd'], rot=30)
        ax.axhline(0, color='black', linewidth=1)
        ax.set_title('REBUS vs baseline deltas (higher is better)')
        ax.set_ylabel('delta')
        ax.grid(axis='y', alpha=0.25)
        plt.tight_layout()
        plt.show()


## 5. Outcome Composition: Success vs Failure

This is the cleanest baseline-compatible view of whether REBUS improves success without increasing collision/deadlock.


In [ ]:
if comparison_df.empty or not set(OUTCOME_COLUMNS).issubset(comparison_df.columns):
    print('Outcome-rate metrics not available.')
else:
    outcome_means = comparison_df.groupby('variant_label', observed=True)[OUTCOME_COLUMNS].mean()
    outcome_means['TimeoutOrOtherFailureRate'] = (
        1.0 - outcome_means['SuccessRate'] - outcome_means['CollisionRate'] - outcome_means['DeadlockRate']
    ).clip(lower=0.0)
    display(outcome_means)

    cols = ['SuccessRate', 'TimeoutOrOtherFailureRate', 'DeadlockRate', 'CollisionRate']
    colors = ['#2ca02c', '#bdbdbd', '#ff7f0e', '#d62728']
    ax = outcome_means[cols].plot(kind='bar', stacked=True, figsize=(9, 5), color=colors, rot=15)
    ax.set_title('Outcome composition: REBUS vs baseline')
    ax.set_ylabel('mean rate')
    ax.set_ylim(0, 1)
    ax.legend(['success', 'timeout/other failure', 'deadlock', 'collision'], bbox_to_anchor=(1.0, 0.5), loc='center left')
    plt.tight_layout()
    plt.show()

    baseline_label = RUN_LABELS[BASELINE_VARIANT]
    rebus_label = RUN_LABELS[REBUS_VARIANT]
    if baseline_label in outcome_means.index and rebus_label in outcome_means.index:
        out_delta = pd.Series({
            'success gain': outcome_means.loc[rebus_label, 'SuccessRate'] - outcome_means.loc[baseline_label, 'SuccessRate'],
            'collision reduction': outcome_means.loc[baseline_label, 'CollisionRate'] - outcome_means.loc[rebus_label, 'CollisionRate'],
            'deadlock reduction': outcome_means.loc[baseline_label, 'DeadlockRate'] - outcome_means.loc[rebus_label, 'DeadlockRate'],
            'collision+deadlock reduction': (
                outcome_means.loc[baseline_label, 'CollisionRate'] + outcome_means.loc[baseline_label, 'DeadlockRate']
                - outcome_means.loc[rebus_label, 'CollisionRate'] - outcome_means.loc[rebus_label, 'DeadlockRate']
            ),
        })
        display(out_delta.rename('REBUS_vs_baseline').to_frame())
        ax = out_delta[['success gain', 'collision reduction', 'deadlock reduction']].plot(
            kind='bar', figsize=(8, 4), color=['#2ca02c', '#d62728', '#ff7f0e'], rot=20
        )
        ax.axhline(0, color='black', linewidth=1)
        ax.set_title('Outcome tradeoff: REBUS vs baseline (higher is better)')
        ax.set_ylabel('rate delta')
        ax.grid(axis='y', alpha=0.25)
        plt.tight_layout()
        plt.show()
        print(
            f"{rebus_label} vs {baseline_label}: "
            f"success {out_delta['success gain']:+.3f}, "
            f"collision reduction {out_delta['collision reduction']:+.3f}, "
            f"deadlock reduction {out_delta['deadlock reduction']:+.3f}."
        )


## 6. Flatten Scenario and Step-Level Data


In [ ]:
def flatten_analysis(payloads, variants):
    scenario_rows = []
    step_rows = []
    for (variant, seed), payload in payloads.items():
        if variant not in variants:
            continue
        label = RUN_LABELS.get(variant, variant)
        for scenario_idx, scenario in enumerate(payload.get('scenario_summaries', [])):
            action_trace = scenario.get('action_trace', []) or []
            n = len(action_trace)
            scenario_rows.append({
                'variant': variant,
                'variant_label': label,
                'seed': seed,
                'scenario_idx': scenario_idx,
                'scenario_family': scenario.get('scenario_family'),
                'partner_style': scenario.get('partner_style'),
                'context_tag': scenario.get('context_tag'),
                'outcome': scenario.get('outcome'),
                'steps': n,
                'belief_confidence_peak': scenario.get('belief_confidence_peak'),
                'belief_shift_moment': scenario.get('belief_shift_moment'),
                'action_switch_moment': scenario.get('action_switch_moment'),
                'context_sensitive_action_regret': scenario.get('context_sensitive_action_regret'),
            })
            tag_set = scenario.get('context_tag_set') or {}
            def trace(name, default=np.nan):
                values = scenario.get(name, []) or []
                return values + [default] * max(0, n - len(values))
            traces = {
                'action_label': trace('action_label_trace', None),
                'belief_confidence': trace('belief_confidence_trace'),
                'belief_entropy': trace('belief_entropy_trace'),
                'belief_class': trace('belief_class_trace'),
                'action_confidence_margin': trace('action_confidence_margin_trace'),
                'context': trace('context_trace', None),
                'evidence_released': trace('evidence_released_trace'),
            }
            for i in range(n):
                row = {
                    'variant': variant,
                    'variant_label': label,
                    'seed': seed,
                    'scenario_idx': scenario_idx,
                    'step': i + 1,
                    'scenario_family': scenario.get('scenario_family'),
                    'partner_style': scenario.get('partner_style'),
                    'outcome': scenario.get('outcome'),
                    'context_sensitive_action_regret': scenario.get('context_sensitive_action_regret'),
                    'urgency': tag_set.get('urgency'),
                    'norm': tag_set.get('norm'),
                    'margin': tag_set.get('margin'),
                    'timeout_pressure': tag_set.get('timeout_pressure'),
                }
                for name, values in traces.items():
                    row[name] = values[i] if i < len(values) else np.nan
                step_rows.append(row)
    return pd.DataFrame(scenario_rows), pd.DataFrame(step_rows)

scenario_df, step_df = flatten_analysis(analysis_payloads, set(COMPARISON_VARIANTS))
print('scenario rows=', len(scenario_df), 'step rows=', len(step_df))
display(scenario_df.head())
display(step_df.head())


## 7. Outcome by Scenario Family and Partner Style


In [ ]:
if scenario_df.empty:
    print('No scenario-level analysis loaded.')
else:
    for group_col in ['scenario_family', 'partner_style']:
        print('\nOutcome rates by', group_col)
        table = pd.crosstab(
            [scenario_df[group_col], scenario_df['variant_label']],
            scenario_df['outcome'],
            normalize='index',
        ).fillna(0)
        display(table)

        success = scenario_df.assign(success=(scenario_df['outcome'] == 'success').astype(float))
        success_pivot = success.pivot_table(index=group_col, columns='variant_label', values='success', aggfunc='mean')
        display(success_pivot)
        ax = success_pivot.plot(kind='bar', figsize=(10, 4), rot=30)
        ax.set_title(f'Success rate by {group_col}')
        ax.set_ylabel('success rate')
        ax.set_ylim(0, 1)
        ax.grid(axis='y', alpha=0.25)
        plt.tight_layout()
        plt.show()


## 8. Belief Confidence and Degradation

Baseline-compatible confidence traces. This cannot show REBUS-specific activation, but it can show whether REBUS changes confidence/degradation patterns.


In [ ]:
def summarize_by_step(df, value_col='belief_confidence'):
    use = df.dropna(subset=[value_col]).copy()
    if use.empty:
        return pd.DataFrame()
    grouped = use.groupby(['variant_label', 'step'], observed=True)[value_col]
    out = grouped.agg(['mean', 'std', 'count']).reset_index()
    out['se'] = out['std'] / np.sqrt(out['count'].clip(lower=1))
    return out

if step_df.empty:
    print('No step-level data loaded.')
else:
    summary = summarize_by_step(step_df, 'belief_confidence')
    display(summary.head())
    plt.figure(figsize=(12, 5))
    for label, sub in summary.groupby('variant_label', observed=True):
        sub = sub.sort_values('step')
        plt.plot(sub['step'], sub['mean'], marker='o', label=label)
        plt.fill_between(sub['step'], sub['mean'] - sub['se'], sub['mean'] + sub['se'], alpha=0.15)
    plt.title('Mean belief confidence by step: REBUS vs baseline')
    plt.xlabel('step')
    plt.ylabel('mean belief confidence')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    degradation_rows = []
    for (label, seed, scenario_idx), sub in step_df.groupby(['variant_label', 'seed', 'scenario_idx'], observed=True):
        sub = sub.sort_values('step')
        conf = pd.to_numeric(sub['belief_confidence'], errors='coerce').dropna()
        if conf.empty:
            continue
        early = conf[sub.loc[conf.index, 'step'] <= 7].mean() if (sub.loc[conf.index, 'step'] <= 7).any() else np.nan
        late = conf[sub.loc[conf.index, 'step'] >= 15].mean() if (sub.loc[conf.index, 'step'] >= 15).any() else np.nan
        peak = conf.max()
        final = conf.iloc[-1]
        degradation_rows.append({
            'variant_label': label,
            'seed': seed,
            'scenario_idx': scenario_idx,
            'scenario_family': sub['scenario_family'].iloc[0],
            'outcome': sub['outcome'].iloc[0],
            'early_mean_confidence': early,
            'late_mean_confidence': late,
            'peak_confidence': peak,
            'final_confidence': final,
            'peak_to_final_drop': peak - final,
            'late_minus_early': late - early if pd.notna(early) and pd.notna(late) else np.nan,
        })
    degradation_df = pd.DataFrame(degradation_rows)
    print('Degradation summary')
    display(degradation_df.groupby('variant_label', observed=True)[['peak_to_final_drop', 'late_minus_early', 'final_confidence']].mean())

    for metric in ['peak_to_final_drop', 'late_minus_early', 'final_confidence']:
        ax = degradation_df.boxplot(column=metric, by='variant_label', figsize=(8, 4), grid=False)
        ax.set_title(metric)
        ax.set_xlabel('')
        plt.suptitle('')
        plt.xticks(rotation=15)
        plt.tight_layout()
        plt.show()


## 9. Action and Context Patterns


In [ ]:
if step_df.empty:
    print('No step-level data loaded.')
else:
    for col in ['action_label', 'context']:
        print('\nRates by', col)
        rates = pd.crosstab(step_df['variant_label'], step_df[col], normalize='index').fillna(0)
        display(rates)
        ax = rates.plot(kind='bar', figsize=(11, 4), rot=15)
        ax.set_title(f'{col} distribution: REBUS vs baseline')
        ax.set_ylabel('step share')
        ax.grid(axis='y', alpha=0.25)
        plt.tight_layout()
        plt.show()

    regret = scenario_df.groupby('variant_label', observed=True)['context_sensitive_action_regret'].mean().rename('mean_context_sensitive_action_regret')
    display(regret.to_frame())
    ax = regret.plot(kind='bar', figsize=(6, 4), color='#9467bd', rot=15)
    ax.set_title('Context-sensitive action regret')
    ax.set_ylabel('mean regret')
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()


## 10. Per-Seed Paired View


In [ ]:
if comparison_df.empty:
    print('No comparable metrics loaded.')
else:
    paired = comparison_df.pivot_table(index='seed', columns='variant_label', values=['SuccessRate', 'CollisionRate', 'DeadlockRate', 'ToMCoordScore'], aggfunc='mean')
    display(paired)

    baseline_label = RUN_LABELS[BASELINE_VARIANT]
    rebus_label = RUN_LABELS[REBUS_VARIANT]
    if baseline_label in paired.columns.get_level_values(1) and rebus_label in paired.columns.get_level_values(1):
        paired_delta = pd.DataFrame(index=paired.index)
        for metric in ['SuccessRate', 'CollisionRate', 'DeadlockRate', 'ToMCoordScore']:
            paired_delta[metric] = paired[(metric, rebus_label)] - paired[(metric, baseline_label)]
        paired_delta['CollisionReduction'] = -paired_delta['CollisionRate']
        paired_delta['DeadlockReduction'] = -paired_delta['DeadlockRate']
        display(paired_delta)

        ax = paired_delta[['SuccessRate', 'CollisionReduction', 'DeadlockReduction', 'ToMCoordScore']].plot(
            kind='bar', figsize=(10, 4), rot=0
        )
        ax.axhline(0, color='black', linewidth=1)
        ax.set_title('Per-seed REBUS vs baseline deltas (higher is better except raw rates renamed)')
        ax.set_ylabel('delta')
        ax.grid(axis='y', alpha=0.25)
        plt.tight_layout()
        plt.show()


## 11. Notes

- REBUS-only metrics are intentionally excluded from the direct baseline comparison because the baseline does not emit them.
- Shared readouts used here are available for both systems: headline metrics, outcomes, scenario-family/partner-style outcomes, action/context distributions, belief confidence, and regret.
- When the resolution run is downloaded, rerun from the top. The notebook will only switch the selected REBUS variant if that run has a higher mean `ToMCoordScore` than the current explicit-mask REBUS variant.
